# DD2360 Assignment 2
Runtime → Change runtime type → **T4 GPU** before running. All build output goes to `build/`.

In [26]:
%cd /content
import os

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
REPO = "tattack99/DD2360-assignment-2"
REPO_DIR = "/content/repo"

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        from google.colab import userdata
        token = userdata.get("GH_TOKEN_DD2360_ASSIGNMENT_2")
        !git clone -q https://{token}@github.com/{REPO}.git {REPO_DIR}
    %cd {REPO_DIR}
    !git pull -q

/content
/content/repo


In [28]:
# Check that a GPU is attached and nvcc is available
!nvidia-smi
!nvcc --version

Wed Nov 18 20:04:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Q1 - 1D Convolution

In [29]:
# Quick check: both kernels should print PASS
!make run

nvcc -arch=native -DTILE_WIDTH=256 -o build/hw2-convolution_tile256 assignment/hw2-convolution.cu
./build/hw2-convolution_tile256 1024
Timing - Allocated host memory                                          6 us
Timing - Allocated device memory                                   285048 us
Timing - Initialize arrays with values                                 37 us
Timing - Copying data to the GPU                                       77 us
Timing - Finished 1D convolution(basic)                               138 us
P[0] = 1.233154
P[1] = 1.405636
P[2] = 1.397376
P[3] = 1.596605
P[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         39 us
Timing - Finished 1D convolution(tiled)                                34 us
P_tiled[0] = 1.233154
P_tiled[1] = 1.405636
P_tiled[2] = 1.397376
P_tiled[3] = 1.596605
P_tiled[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         31 us
Basic kernel vs CPU: PASS (0 mismatches)
Tiled kernel vs CP

In [30]:
# Q1.3 tile size: run every tile size on the largest input, compare the tiled kernel times
!make tilesweep ARGS=16384

=== TILE = 32 ===
nvcc -arch=native -DTILE_WIDTH=32 -o build/hw2-convolution_tile32 assignment/hw2-convolution.cu
./build/hw2-convolution_tile32 16384
Timing - Allocated host memory                                          7 us
Timing - Allocated device memory                                   281239 us
Timing - Initialize arrays with values                                497 us
Timing - Copying data to the GPU                                       92 us
Timing - Finished 1D convolution(basic)                               144 us
P[0] = 1.233154
P[1] = 1.405636
P[2] = 1.397376
P[3] = 1.596605
P[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         56 us
Timing - Finished 1D convolution(tiled)                                40 us
P_tiled[0] = 1.233154
P_tiled[1] = 1.405636
P_tiled[2] = 1.397376
P_tiled[3] = 1.596605
P_tiled[4] = 1.297674
Timing - Copying output P to the CPU and print out the results        121 us
Basic kernel vs CPU: PASS (0 mismatches)
Ti

In [31]:
# Q1.3 screenshot: N = 1024 ... 16384 with the chosen tile size
TILE = 256
!make sweep TILE={TILE}

=== N = 1024, TILE = 256 ===
Timing - Allocated host memory                                          7 us
Timing - Allocated device memory                                   285778 us
Timing - Initialize arrays with values                                 31 us
Timing - Copying data to the GPU                                       41 us
Timing - Finished 1D convolution(basic)                               141 us
P[0] = 1.233154
P[1] = 1.405636
P[2] = 1.397376
P[3] = 1.596605
P[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         38 us
Timing - Finished 1D convolution(tiled)                                36 us
P_tiled[0] = 1.233154
P_tiled[1] = 1.405636
P_tiled[2] = 1.397376
P_tiled[3] = 1.596605
P_tiled[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         33 us
Basic kernel vs CPU: PASS (0 mismatches)
Tiled kernel vs CPU: PASS (0 mismatches)
Timing - Free memory resources                                        128 us
=== N 

In [32]:
# Q1.4 Nsight Systems: where the time goes
!make profile TILE={TILE} ARGS=16384

nsys profile --stats=true --force-overwrite true -o build/hw2-convolution-report ./build/hw2-convolution_tile256 16384
Timing - Allocated host memory                                         11 us
Timing - Allocated device memory                                   534250 us
Timing - Initialize arrays with values                                416 us
Timing - Copying data to the GPU                                       98 us
Timing - Finished 1D convolution(basic)                               188 us
P[0] = 1.233154
P[1] = 1.405636
P[2] = 1.397376
P[3] = 1.596605
P[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         56 us
Timing - Finished 1D convolution(tiled)                                50 us
P_tiled[0] = 1.233154
P_tiled[1] = 1.405636
P_tiled[2] = 1.397376
P_tiled[3] = 1.596605
P_tiled[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         78 us
Basic kernel vs CPU: PASS (0 mismatches)
Tiled kernel vs CPU: PASS (0 misma

In [33]:
# Q1.4 Nsight Compute: occupancy, shared memory, memory throughput
!make ncu TILE={TILE} ARGS=16384

ncu --section SpeedOfLight --section MemoryWorkloadAnalysis --section Occupancy --section LaunchStats ./build/hw2-convolution_tile256 16384
Timing - Allocated host memory                                          6 us
==PROF== Connected to process 12319 (/content/repo/build/hw2-convolution_tile256)
Timing - Allocated device memory                                   556311 us
Timing - Initialize arrays with values                                577 us
Timing - Copying data to the GPU                                       74 us
==PROF== Profiling "convolution_1D_basic" - 0: 0%....50%....100% - 9 passes
Timing - Finished 1D convolution(basic)                           2930668 us
P[0] = 1.233154
P[1] = 1.405636
P[2] = 1.397376
P[3] = 1.596605
P[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         86 us
==PROF== Profiling "convolution_1D_tiled" - 1: 0%....50%....100% - 9 passes
Timing - Finished 1D convolution(tiled)                           1852466 us
P_tiled

# Question 3

In [34]:
%cd /content/repo/llama2.c/


/content/repo/llama2.c


In [35]:
#We made a modified run.c to include a counter of matmul calls.
!gcc -Ofast -o run my_run.c -lm

In [36]:
model = "stories15M" #We want the small model only
download_url = "https://huggingface.co/karpathy/tinyllamas/resolve/main/stories15M.bin"
!wget $download_url
model_file = model + ".bin"

--2026-11-18 20:06:11--  https://huggingface.co/karpathy/tinyllamas/resolve/main/stories15M.bin
Resolving huggingface.co (huggingface.co)... 99.86.101.64, 99.86.101.56, 99.86.101.39, ...
Connecting to huggingface.co (huggingface.co)|99.86.101.64|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/64c19eb0275dc8cd7f170b8f/07dbdd97d4def5722a8deaeb97073a464e041687313cbf6b06cccb9efdd0c4b7?X-Xet-Cas-Uid=public&xip=CEIgaS0hlXo&response-content-type=application%2Foctet-stream&user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27stories15M.bin%3B+filename%3D%22stories15M.bin%22%3B&Expires=1795035971&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjRjMTllYjAyNzVkYzhjZDdmMTcwYjhmLzA3ZGJkZDk3ZDRkZWY1NzIyYThkZWFlYjk3MDczYTQ2NGUwNDE2ODczMTNjYmY2YjA2Y2NjYjllZmRkMGM0YjdcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZ4aXA9Q0VJZ2FTMGhsWG8mcmVzcG9uc2UtY29udGVudC10eXBlPSomdXNlcl9pZD1w

In [37]:
#@title Generate Stories

# Generate args
max_token = 256 #@param {type:"slider", min:32, max:1024, step:32}
temperature = 0.8 #@param {type:"slider", min:0.0, max:1, step:0.05}
top_p = 0.9 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
prompt = "One day, Lily met a Shoggoth" #@param {type:"string"}

print(f"model: {model_file}, max_token: {max_token}, temperature: {temperature}, top_p: {top_p}, prompt: {prompt}")
print(f"----------------------------\n")

cmd = f'./run {model_file} -t {temperature} -p {top_p} -n {max_token} -i "{prompt}"'
!{cmd}

model: stories15M.bin, max_token: 256, temperature: 0.8, top_p: 0.9, prompt: One day, Lily met a Shoggoth
----------------------------

One day, Lily met a Shoggoth in the park. She was very happy to see it and she wanted to give it a name. She picked it up and said, "This is a good name!"
Just then, a boy came over and said, "That's a nice name. Can I have it?" Lily was hesitant, but the boy seemed nice. He said, "I'll give you this shirt." 
Lily was so excited! She thought it was very thoughtful of the boy to give her a shirt. She said, "Thank you!" and gave him a hug.
The boy smiled and said, "You're welcome!" Lily smiled back and said, "Thank you!"
achieved tok/s: 125.496426
matmul calls: 6837


#Porting to GPU


In [41]:
!make runcuda_1

gcc -O3 -c runcuda.c -o runcuda.o
nvcc -O3 -c matmul.cu -o matmul.o
nvcc -O3 runcuda.o matmul.o -o runcuda_1 -lm


In [43]:
!./runcuda_1 -n 256 -i "Once in a dark forest"

Usage:   run <checkpoint> [options]
Example: run model.bin -n 256 -i "Once upon a time"
Options:
  -t <float>  temperature in [0,inf], default 1.0
  -p <float>  p value in top-p (nucleus) sampling in [0,1] default 0.9
  -s <int>    random seed, default time(NULL)
  -n <int>    number of steps to run for, default 256. 0 = max_seq_len
  -i <string> input prompt
  -z <string> optional path to custom tokenizer
  -m <string> mode: generate|chat, default: generate
  -y <string> (optional) system prompt in chat mode


In [ ]:
#@title Run Meta's Llama 2 models - Not sure we do this...

#@markdown input your huggingface [access token](https://huggingface.co/settings/tokens) to download Meta's Llama 2 models.

from huggingface_hub import snapshot_download

token = "replace your huggingface access token" #@param {type:"string"}
path = snapshot_download(repo_id="meta-llama/Llama-2-7b",cache_dir="Llama-2-7b", use_auth_token=token)

!python export.py llama2_7b.bin --meta-llama $path

print("./run llama2_7b.bin\n")
!./run llama2_7b.bin

TypeError: snapshot_download() got an unexpected keyword argument 'use_auth_token'